# Evaluate LoRA - Old Photo Repair Inpainting

This notebook performs the report-oriented evaluation for the old photo repair LoRA experiment.

It compares:

- `damaged`: damaged input image
- `pretrained_raw`: raw output from `runwayml/stable-diffusion-inpainting`
- `pretrained_hard_composite`: pretrained output inside mask, damaged outside mask
- `lora_raw`: raw output from pretrained model + old-photo LoRA
- `lora_hard_composite`: LoRA output inside mask, damaged outside mask

Metrics:

- Full image: PSNR, SSIM, LPIPS
- Mask region: MSE, MAE, PSNR
- Runtime per sample for model-generated methods

## 0. Setup

Recommended Kaggle settings:

- Accelerator: GPU
- Internet: ON
- Add the processed dataset as input: https://www.kaggle.com/datasets/dwctien/openphoto-restore-filtered-inpainting-subset
- Add LoRA checkpoint weights from the training notebook output, or use the published weights dataset: https://www.kaggle.com/datasets/dwctien/old-photo-lora-weights

Before the full report evaluation, this notebook first benchmarks all uploaded LoRA checkpoints on a small fixed subset, selects the best checkpoint, then runs the full evaluation with that selected checkpoint.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors pandas pillow tqdm matplotlib scikit-image lpips

In [ ]:
from pathlib import Path
import json
import random
import shutil
import time

import lpips
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFilter
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tqdm.auto import tqdm

from diffusers import StableDiffusionInpaintPipeline

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Configuration

In [ ]:
DATASET_SLUG = 'openphoto-restore-filtered-inpainting-subset'
DATASET_ROOT = None
for candidate in [Path('/kaggle/input') / DATASET_SLUG]:
    if (candidate / 'metadata_all.csv').exists():
        DATASET_ROOT = candidate
        break

if DATASET_ROOT is None:
    matches = list(Path('/kaggle/input').glob('**/metadata_all.csv'))
    assert matches, 'Dataset not found. Add openphoto-restore-filtered-inpainting-subset as Kaggle input.'
    DATASET_ROOT = matches[0].parent

LORA_CHECKPOINT_SLUG = 'old-photo-lora-weights'
LORA_CHECKPOINT_ROOT = None
for candidate in [
    Path('/kaggle/input') / LORA_CHECKPOINT_SLUG,
    Path('/kaggle/input/datasets/dwctien') / LORA_CHECKPOINT_SLUG,
    Path('/kaggle/working/outputs_old_photo_lora_v2/checkpoints'),
]:
    if (candidate / 'step_200' / 'pytorch_lora_weights.safetensors').exists():
        LORA_CHECKPOINT_ROOT = candidate
        break

assert LORA_CHECKPOINT_ROOT is not None, 'LoRA checkpoints not found. Add old-photo-lora-weights as input or run the training notebook first.'
LORA_CHECKPOINTS = {
    'step_200': LORA_CHECKPOINT_ROOT / 'step_200',
    'step_400': LORA_CHECKPOINT_ROOT / 'step_400',
    'step_600': LORA_CHECKPOINT_ROOT / 'step_600',
    'step_800': LORA_CHECKPOINT_ROOT / 'step_800',
    'step_1000': LORA_CHECKPOINT_ROOT / 'step_1000',
}
for name, path in LORA_CHECKPOINTS.items():
    weights_path = path / 'pytorch_lora_weights.safetensors'
    assert weights_path.exists(), f'Missing LoRA checkpoint {name}: {weights_path}'

MODEL_ID = 'runwayml/stable-diffusion-inpainting'
PROMPT = 'restore an old damaged photo, realistic photo, clean details'
NEGATIVE_PROMPT = 'blurry, distorted, low quality, artifacts, unrealistic texture'

IMAGE_SIZE = 512
SEED = 42
NUM_TEST_SAMPLES = 100
CHECKPOINT_SELECTION_SAMPLES = 20
NUM_QUALITATIVE_GRIDS = 20
INFERENCE_STEPS = 30
GUIDANCE_SCALE = 7.5

OUTPUT_ROOT = Path('/kaggle/working/outputs_old_photo_evaluation_v2')
GENERATED_DIR = OUTPUT_ROOT / 'generated'
GRID_DIR = OUTPUT_ROOT / 'qualitative_grids'
METRIC_DIR = OUTPUT_ROOT / 'metrics'

for d in [GENERATED_DIR, GRID_DIR, METRIC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
weight_dtype = torch.float16 if device == 'cuda' else torch.float32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Dataset root:', DATASET_ROOT)
print('LoRA checkpoint root:', LORA_CHECKPOINT_ROOT)
print('LoRA checkpoints:', list(LORA_CHECKPOINTS))
print('Output root:', OUTPUT_ROOT)
print('Device:', device, 'dtype:', weight_dtype)

## 2. Load Test Split

In [ ]:
TEST_ROOT = DATASET_ROOT / 'test'
TEST_METADATA_PATH = TEST_ROOT / 'metadata.csv'
assert TEST_METADATA_PATH.exists(), f'Missing test metadata: {TEST_METADATA_PATH}'

test_df = pd.read_csv(TEST_METADATA_PATH)
test_df['damaged_abs'] = test_df['damaged_path'].apply(lambda p: str(DATASET_ROOT / p))
test_df['pristine_abs'] = test_df['pristine_path'].apply(lambda p: str(DATASET_ROOT / p))
test_df['mask_abs'] = test_df['mask_path'].apply(lambda p: str(DATASET_ROOT / p))

for col in ['damaged_abs', 'pristine_abs', 'mask_abs']:
    assert test_df[col].apply(lambda p: Path(p).exists()).all(), f'Missing files in {col}'

EVAL_MIN_MASK_AREA = 0.005
EVAL_MAX_MASK_AREA = 0.25
filtered_test_df = test_df[
    test_df['mask_area'].between(EVAL_MIN_MASK_AREA, EVAL_MAX_MASK_AREA)
].copy().reset_index(drop=True)
assert len(filtered_test_df) > 0, 'No test samples left after evaluation mask filtering.'
eval_df = filtered_test_df.sample(min(NUM_TEST_SAMPLES, len(filtered_test_df)), random_state=SEED).reset_index(drop=True)
checkpoint_selection_df = filtered_test_df.sample(
    min(CHECKPOINT_SELECTION_SAMPLES, len(filtered_test_df)),
    random_state=SEED + 7,
).reset_index(drop=True)

print('Total test samples:', len(test_df))
print('Filtered test samples:', len(filtered_test_df))
print('Evaluation samples:', len(eval_df))
print('Checkpoint selection samples:', len(checkpoint_selection_df))
display(eval_df['mask_area'].describe())

## 3. Utility Functions

In [ ]:
def load_rgb(path):
    return Image.open(path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)

def load_mask(path):
    mask = Image.open(path).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST)
    arr = (np.asarray(mask) > 127).astype(np.uint8) * 255
    return Image.fromarray(arr, mode='L')

def hard_composite(generated, damaged, mask):
    gen = np.asarray(generated.convert('RGB')).astype(np.uint8)
    dam = np.asarray(damaged.convert('RGB')).astype(np.uint8)
    m = np.asarray(mask.convert('L')) > 127
    out = dam.copy()
    out[m] = gen[m]
    return Image.fromarray(out, mode='RGB')

def fuzzy_composite(generated, damaged, mask, blur_radius=4):
    gen = np.asarray(generated.convert('RGB')).astype(np.float32)
    dam = np.asarray(damaged.convert('RGB')).astype(np.float32)
    alpha = np.asarray(mask.convert('L').filter(ImageFilter.GaussianBlur(blur_radius))).astype(np.float32) / 255.0
    alpha = alpha[..., None]
    out = gen * alpha + dam * (1.0 - alpha)
    return Image.fromarray(out.clip(0, 255).astype(np.uint8), mode='RGB')

def pil_to_float(image):
    return np.asarray(image.convert('RGB')).astype(np.float32) / 255.0

def lpips_tensor(image):
    arr = pil_to_float(image)
    tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    return tensor.to(device=device, dtype=torch.float32) * 2.0 - 1.0

lpips_model = lpips.LPIPS(net='alex').to(device).eval()

def compute_full_metrics(pred, target):
    pred_arr = pil_to_float(pred)
    tgt_arr = pil_to_float(target)
    with torch.no_grad():
        lpips_value = float(lpips_model(lpips_tensor(pred), lpips_tensor(target)).item())
    return {
        'psnr': peak_signal_noise_ratio(tgt_arr, pred_arr, data_range=1.0),
        'ssim': structural_similarity(tgt_arr, pred_arr, channel_axis=2, data_range=1.0),
        'lpips': lpips_value,
    }

def compute_mask_metrics(pred, target, mask):
    pred_arr = pil_to_float(pred)
    tgt_arr = pil_to_float(target)
    mask_arr = np.asarray(mask.convert('L')) > 127
    if mask_arr.sum() == 0:
        return {'mask_mse': np.nan, 'mask_mae': np.nan, 'mask_psnr': np.nan}
    diff = pred_arr[mask_arr] - tgt_arr[mask_arr]
    mse = float(np.mean(diff ** 2))
    mae = float(np.mean(np.abs(diff)))
    mask_psnr = float('inf') if mse == 0 else float(10.0 * np.log10(1.0 / mse))
    return {'mask_mse': mse, 'mask_mae': mae, 'mask_psnr': mask_psnr}

def make_grid(panels):
    grid = Image.new('RGB', (IMAGE_SIZE * len(panels), IMAGE_SIZE), 'white')
    for i, panel in enumerate(panels):
        grid.paste(panel.convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE)), (i * IMAGE_SIZE, 0))
    return grid

def add_metric_row(rows, sample_id, method, image, target, mask, runtime_sec=None):
    rows.append({
        'sample_id': sample_id,
        'method': method,
        'runtime_sec': runtime_sec,
        **compute_full_metrics(image, target),
        **compute_mask_metrics(image, target, mask),
    })

## 4. Select Best LoRA Checkpoint

This stage evaluates all uploaded LoRA checkpoints on a small fixed subset. The selected checkpoint is the one with the best `fuzzy_psnr_mean`, with `fuzzy_ssim_mean` as the tie-breaker.

The full report evaluation below uses only this selected checkpoint.

In [ ]:
@torch.no_grad()
def benchmark_lora_checkpoint(checkpoint_name, checkpoint_root, rows_df):
    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=weight_dtype,
        safety_checker=None,
    ).to(device)
    pipe.load_lora_weights(str(checkpoint_root))
    pipe.enable_attention_slicing()
    if hasattr(pipe, 'enable_vae_slicing'):
        pipe.enable_vae_slicing()

    rows = []
    for i, row in tqdm(rows_df.iterrows(), total=len(rows_df), desc=f'Benchmark {checkpoint_name}'):
        sample_id = int(row['id'])
        pristine = load_rgb(row['pristine_abs'])
        damaged = load_rgb(row['damaged_abs'])
        mask = load_mask(row['mask_abs'])
        generator = torch.Generator(device=device).manual_seed(SEED + 1000 + i)

        start = time.time()
        raw = pipe(
            prompt=PROMPT,
            negative_prompt=NEGATIVE_PROMPT,
            image=damaged,
            mask_image=mask,
            num_inference_steps=INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator,
        ).images[0]
        runtime_sec = time.time() - start

        hard = hard_composite(raw, damaged, mask)
        fuzzy = fuzzy_composite(raw, damaged, mask)
        hard_full = compute_full_metrics(hard, pristine)
        fuzzy_full = compute_full_metrics(fuzzy, pristine)
        hard_mask = compute_mask_metrics(hard, pristine, mask)
        fuzzy_mask = compute_mask_metrics(fuzzy, pristine, mask)

        rows.append({
            'checkpoint': checkpoint_name,
            'sample_id': sample_id,
            'runtime_sec': runtime_sec,
            'hard_psnr': hard_full['psnr'],
            'hard_ssim': hard_full['ssim'],
            'hard_lpips': hard_full['lpips'],
            'hard_mask_psnr': hard_mask['mask_psnr'],
            'fuzzy_psnr': fuzzy_full['psnr'],
            'fuzzy_ssim': fuzzy_full['ssim'],
            'fuzzy_lpips': fuzzy_full['lpips'],
            'fuzzy_mask_psnr': fuzzy_mask['mask_psnr'],
        })

    del pipe
    torch.cuda.empty_cache()
    return pd.DataFrame(rows)


checkpoint_metric_frames = []
for checkpoint_name, checkpoint_root in LORA_CHECKPOINTS.items():
    checkpoint_metric_frames.append(
        benchmark_lora_checkpoint(checkpoint_name, checkpoint_root, checkpoint_selection_df)
    )

checkpoint_metrics_df = pd.concat(checkpoint_metric_frames, ignore_index=True)
checkpoint_summary_df = checkpoint_metrics_df.groupby('checkpoint').agg(
    hard_psnr_mean=('hard_psnr', 'mean'),
    hard_ssim_mean=('hard_ssim', 'mean'),
    hard_lpips_mean=('hard_lpips', 'mean'),
    hard_mask_psnr_mean=('hard_mask_psnr', 'mean'),
    fuzzy_psnr_mean=('fuzzy_psnr', 'mean'),
    fuzzy_ssim_mean=('fuzzy_ssim', 'mean'),
    fuzzy_lpips_mean=('fuzzy_lpips', 'mean'),
    fuzzy_mask_psnr_mean=('fuzzy_mask_psnr', 'mean'),
    runtime_sec_mean=('runtime_sec', 'mean'),
).sort_values(['fuzzy_psnr_mean', 'fuzzy_ssim_mean'], ascending=[False, False])

checkpoint_metrics_df.to_csv(METRIC_DIR / 'checkpoint_selection_metrics_per_sample.csv', index=False)
checkpoint_summary_df.to_csv(METRIC_DIR / 'checkpoint_selection_summary.csv')

SELECTED_LORA_NAME = checkpoint_summary_df.index[0]
SELECTED_LORA_ROOT = LORA_CHECKPOINTS[SELECTED_LORA_NAME]

display(checkpoint_summary_df)
print('Selected LoRA checkpoint:', SELECTED_LORA_NAME)
print('Selected LoRA root:', SELECTED_LORA_ROOT)

## 5. Generate Pretrained Outputs

In [ ]:
@torch.no_grad()
def generate_method_outputs(method_name, lora_root=None):
    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=weight_dtype,
        safety_checker=None,
    ).to(device)
    if lora_root is not None:
        pipe.load_lora_weights(str(lora_root))
    pipe.enable_attention_slicing()
    if hasattr(pipe, 'enable_vae_slicing'):
        pipe.enable_vae_slicing()

    method_dir = GENERATED_DIR / method_name
    method_dir.mkdir(parents=True, exist_ok=True)

    runtime_rows = []
    for i, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=method_name):
        damaged = load_rgb(row['damaged_abs'])
        mask = load_mask(row['mask_abs'])
        generator = torch.Generator(device=device).manual_seed(SEED + i)

        start = time.time()
        pred = pipe(
            prompt=PROMPT,
            negative_prompt=NEGATIVE_PROMPT,
            image=damaged,
            mask_image=mask,
            num_inference_steps=INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator,
        ).images[0]
        runtime_sec = time.time() - start

        sample_id = int(row['id'])
        pred.save(method_dir / f'{sample_id:06d}_raw.png')
        hard_composite(pred, damaged, mask).save(method_dir / f'{sample_id:06d}_hard.png')
        fuzzy_composite(pred, damaged, mask).save(method_dir / f'{sample_id:06d}_fuzzy.png')
        runtime_rows.append({'sample_id': sample_id, 'method': method_name, 'runtime_sec': runtime_sec})

    del pipe
    torch.cuda.empty_cache()
    return pd.DataFrame(runtime_rows)

pretrained_runtime = generate_method_outputs('pretrained')
display(pretrained_runtime['runtime_sec'].describe())

## 6. Generate Selected LoRA Outputs

In [ ]:
lora_runtime = generate_method_outputs('lora', lora_root=SELECTED_LORA_ROOT)
display(lora_runtime['runtime_sec'].describe())

## 7. Compute Metrics

In [ ]:
runtime_lookup = pd.concat([pretrained_runtime, lora_runtime], ignore_index=True)
runtime_lookup = runtime_lookup.set_index(['method', 'sample_id'])['runtime_sec'].to_dict()

rows = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc='Metrics'):
    sample_id = int(row['id'])
    pristine = load_rgb(row['pristine_abs'])
    damaged = load_rgb(row['damaged_abs'])
    mask = load_mask(row['mask_abs'])

    add_metric_row(rows, sample_id, 'damaged', damaged, pristine, mask, runtime_sec=0.0)

    for method in ['pretrained', 'lora']:
        method_dir = GENERATED_DIR / method
        runtime_sec = runtime_lookup.get((method, sample_id))
        raw = Image.open(method_dir / f'{sample_id:06d}_raw.png').convert('RGB')
        hard = Image.open(method_dir / f'{sample_id:06d}_hard.png').convert('RGB')
        fuzzy = Image.open(method_dir / f'{sample_id:06d}_fuzzy.png').convert('RGB')

        add_metric_row(rows, sample_id, f'{method}_raw', raw, pristine, mask, runtime_sec=runtime_sec)
        add_metric_row(rows, sample_id, f'{method}_hard_composite', hard, pristine, mask, runtime_sec=runtime_sec)
        add_metric_row(rows, sample_id, f'{method}_fuzzy_composite', fuzzy, pristine, mask, runtime_sec=runtime_sec)

metrics_df = pd.DataFrame(rows)
summary_df = metrics_df.groupby('method').agg(
    psnr_mean=('psnr', 'mean'),
    psnr_std=('psnr', 'std'),
    ssim_mean=('ssim', 'mean'),
    ssim_std=('ssim', 'std'),
    lpips_mean=('lpips', 'mean'),
    lpips_std=('lpips', 'std'),
    mask_mse_mean=('mask_mse', 'mean'),
    mask_mae_mean=('mask_mae', 'mean'),
    mask_psnr_mean=('mask_psnr', 'mean'),
    runtime_sec_mean=('runtime_sec', 'mean'),
).sort_index()

metrics_df.to_csv(METRIC_DIR / 'old_photo_eval_metrics_per_sample.csv', index=False)
summary_df.to_csv(METRIC_DIR / 'old_photo_eval_summary.csv')

display(summary_df)

## 8. Qualitative Grids

Panel order: `pristine | damaged | mask | pretrained hard | LoRA hard`.

In [ ]:
for i, row in eval_df.head(NUM_QUALITATIVE_GRIDS).iterrows():
    sample_id = int(row['id'])
    pristine = load_rgb(row['pristine_abs'])
    damaged = load_rgb(row['damaged_abs'])
    mask = load_mask(row['mask_abs'])
    pretrained_hard = Image.open(GENERATED_DIR / 'pretrained' / f'{sample_id:06d}_hard.png').convert('RGB')
    lora_hard = Image.open(GENERATED_DIR / 'lora' / f'{sample_id:06d}_hard.png').convert('RGB')
    grid = make_grid([pristine, damaged, mask.convert('RGB'), pretrained_hard, lora_hard])
    grid.save(GRID_DIR / f'eval_grid_{sample_id:06d}.png')

sample_grid = sorted(GRID_DIR.glob('*.png'))[0]
print('Saved grids:', GRID_DIR)
print('Example:', sample_grid)
plt.figure(figsize=(20, 4))
plt.imshow(Image.open(sample_grid))
plt.axis('off')
plt.show()

## 9. Save Report Notes And Archive

In [ ]:
best_by_psnr = summary_df['psnr_mean'].idxmax()
best_by_ssim = summary_df['ssim_mean'].idxmax()
best_by_lpips = summary_df['lpips_mean'].idxmin()

notes = {
    'dataset_root': str(DATASET_ROOT),
    'lora_checkpoint_root': str(LORA_CHECKPOINT_ROOT),
    'selected_lora_checkpoint': SELECTED_LORA_NAME,
    'selected_lora_root': str(SELECTED_LORA_ROOT),
    'checkpoint_selection_samples': int(len(checkpoint_selection_df)),
    'num_test_samples': int(len(eval_df)),
    'inference_steps': INFERENCE_STEPS,
    'guidance_scale': GUIDANCE_SCALE,
    'best_by_psnr': best_by_psnr,
    'best_by_ssim': best_by_ssim,
    'best_by_lpips': best_by_lpips,
}

with open(METRIC_DIR / 'report_notes.json', 'w', encoding='utf-8') as f:
    json.dump(notes, f, indent=2)

archive_path = shutil.make_archive('/kaggle/working/old_photo_evaluation_outputs', 'zip', root_dir=OUTPUT_ROOT)

print(json.dumps(notes, indent=2))
print('Per-sample metrics:', METRIC_DIR / 'old_photo_eval_metrics_per_sample.csv')
print('Summary metrics:', METRIC_DIR / 'old_photo_eval_summary.csv')
print('Qualitative grids:', GRID_DIR)
print('Output archive:', archive_path)